# 🐱🐶 Workshop 1 — Supervised Learning: Cat vs Dog Classifier

**AI Frontier · Hands-on session**

In this notebook we teach a computer to tell cats from dogs using **labelled photos**.
That's **supervised learning**: every training image comes with the right answer ("cat" or "dog").

**What we'll do**
1. Download the dataset
2. Look at the data
3. Pre-process the images
4. Build and train a CNN from scratch
5. Improve it with transfer learning
6. Evaluate and test on your own photos

> ⚙️ **Before you start:** `Runtime → Change runtime type → T4 GPU`. Training is ~10× faster on a GPU.

## 1. The dataset

We use **Cats and Dogs (filtered)** — a small, clean version of Kaggle's famous *Dogs vs Cats* dataset, hosted by Google:

| Split | Cats | Dogs |
|---|---|---|
| Train | 1,000 | 1,000 |
| Validation | 500 | 500 |

✅ **Good news: no manual download needed.** The next cell downloads it straight into Colab (~65 MB).

**Want to download it yourself / use the full dataset?**
- Small version (used here): https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip
- Full Kaggle version (25,000 images, needs a free Kaggle account): https://www.kaggle.com/c/dogs-vs-cats/data
- Also available in TensorFlow Datasets as `cats_vs_dogs`: https://www.tensorflow.org/datasets/catalog/cats_vs_dogs

In [ ]:
!wget -q https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip
!unzip -q -o cats_and_dogs_filtered.zip
!ls cats_and_dogs_filtered/train

## 2. Imports and settings

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

BASE_DIR  = "cats_and_dogs_filtered"
TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "validation")

IMG_SIZE   = (160, 160)   # every image gets resized to this
BATCH_SIZE = 32
SEED       = 42

## 3. Look at the data

Always look at your data before training. How many images? What do they look like?

In [ ]:
for split, d in [("train", TRAIN_DIR), ("validation", VAL_DIR)]:
    for cls in ["cats", "dogs"]:
        n = len(os.listdir(os.path.join(d, cls)))
        print(f"{split:<11} {cls}: {n}")

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=True, seed=SEED)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False)

class_names = train_ds.class_names
print("Classes:", class_names)   # ['cats', 'dogs'] -> label 0 = cat, 1 = dog

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[int(labels[i][0])])
        plt.axis("off")
plt.show()

## 4. Pre-processing

Two things happen here:

- **Rescaling** – pixel values go from 0–255 down to 0–1, which makes training stable.
- **Data augmentation** – random flips, rotations and zooms. The model sees a slightly different version of each photo every epoch, so it learns *"what a cat looks like"* instead of memorising 1,000 specific photos.

We also `cache` + `prefetch` so the GPU never waits for data.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="augmentation")

# What does augmentation do? Same image, 9 random versions:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    img = images[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        aug = data_augmentation(tf.expand_dims(img, 0), training=True)
        plt.imshow(aug[0].numpy().astype("uint8"))
        plt.axis("off")
plt.show()

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

## 5. Build a CNN from scratch

A **Convolutional Neural Network** looks at small patches of the image and learns patterns:
edges → textures → ears/eyes/noses → "cat" or "dog".

```
Image → [Conv → Pool] ×3 → Flatten → Dense → Output (0 = cat, 1 = dog)
```

In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=IMG_SIZE + (3,)),
    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(1, activation="sigmoid"),   # probability of "dog"
], name="cnn_from_scratch")

cnn.compile(optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"])
cnn.summary()

## 6. Train it

One **epoch** = the model sees every training image once.
Watch `accuracy` (training) and `val_accuracy` (images it has never trained on). ⏱️ ~2–3 min on a T4 GPU.

In [ ]:
EPOCHS = 15
history = cnn.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:
def plot_history(h, title):
    acc, val_acc = h.history["accuracy"], h.history["val_accuracy"]
    loss, val_loss = h.history["loss"], h.history["val_loss"]
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label="train"); plt.plot(val_acc, label="validation")
    plt.title(f"{title} – accuracy"); plt.xlabel("epoch"); plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(loss, label="train"); plt.plot(val_loss, label="validation")
    plt.title(f"{title} – loss"); plt.xlabel("epoch"); plt.legend()
    plt.show()

plot_history(history, "CNN from scratch")

💬 **Discuss:** You'll probably land around **70–80 %** validation accuracy.
Not bad, but 2,000 photos is very little data to learn "what is a cat" from zero. Can we do better *without* collecting more data?

## 7. Transfer learning — standing on the shoulders of giants

**MobileNetV2** has already been trained on **1.4 million images** (ImageNet). It already knows edges, fur, eyes, shapes.
We **freeze** that knowledge and only train a tiny new "head" that says cat vs dog.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
base_model.trainable = False   # freeze the pre-trained layers

inputs  = keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = layers.Rescaling(1./127.5, offset=-1)(x)   # MobileNetV2 expects pixels in [-1, 1]
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

tl_model = keras.Model(inputs, outputs, name="mobilenet_transfer")
tl_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                 loss="binary_crossentropy",
                 metrics=["accuracy"])

print("Trainable params:", sum(np.prod(v.shape) for v in tl_model.trainable_variables))

In [ ]:
history_tl = tl_model.fit(train_ds, validation_data=val_ds, epochs=5)
plot_history(history_tl, "Transfer learning")

🚀 Usually **~95 %+** in just 5 epochs, with a fraction of the training. That's why almost every real-world vision project starts from a pre-trained model.

## 8. Evaluate — where does it go wrong?

A **confusion matrix** shows how many cats were called dogs and vice-versa.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

model = tl_model   # change to `cnn` to compare

y_true, y_prob = [], []
for images, labels in val_ds:
    y_prob.extend(model.predict(images, verbose=0).ravel())
    y_true.extend(labels.numpy().ravel())
y_true = np.array(y_true).astype(int)
y_pred = (np.array(y_prob) > 0.5).astype(int)

ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=class_names).plot(cmap="Blues")
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Show some images the model got WRONG
wrong = np.where(y_pred != y_true)[0][:9]
val_images = np.concatenate([x.numpy() for x, _ in val_ds])

plt.figure(figsize=(10, 10))
for i, idx in enumerate(wrong):
    plt.subplot(3, 3, i + 1)
    plt.imshow(val_images[idx].astype("uint8"))
    plt.title(f"true: {class_names[y_true[idx]]}\npred: {class_names[y_pred[idx]]} ({y_prob[idx]:.2f})")
    plt.axis("off")
plt.show()

## 9. Try it on your own photo 📸

Run the cell and upload any cat or dog picture (from your phone, Google Images, etc.).

In [ ]:
from google.colab import files

uploaded = files.upload()
for fname in uploaded:
    img = keras.utils.load_img(fname, target_size=IMG_SIZE)
    arr = keras.utils.img_to_array(img)[None, ...]       # shape (1, 160, 160, 3)
    p_dog = float(model.predict(arr, verbose=0)[0][0])
    label = "DOG 🐶" if p_dog > 0.5 else "CAT 🐱"
    conf = p_dog if p_dog > 0.5 else 1 - p_dog
    plt.imshow(img); plt.axis("off")
    plt.title(f"{label}  ({conf:.0%} sure)")
    plt.show()

## 10. Save the model (optional)

In [ ]:
model.save("cat_dog_classifier.keras")
# files.download("cat_dog_classifier.keras")   # uncomment to download to your laptop

## 🧪 Challenges

1. Remove `data_augmentation` from the CNN and retrain. What happens to the gap between train and validation accuracy? (Hint: overfitting.)
2. Change `IMG_SIZE` to `(96, 96)`. Faster? Less accurate?
3. Upload a photo of something that's **neither** a cat nor a dog (a car, yourself). What does the model say, and why?
4. **Fine-tuning:** set `base_model.trainable = True`, recompile with `Adam(1e-5)`, and train 3 more epochs. Does accuracy go up?

### ✅ Recap
- **Supervised learning** = learning from *labelled* examples.
- Pipeline: **data → pre-processing → model → training → evaluation → prediction**.
- **Transfer learning** gets great results with small datasets.